# Answer Set Programming (ASP / clingo)

A self-contained refresher on **Answer Set Programming** — a declarative, model-finding paradigm for combinatorial search and knowledge representation, driven from Python via the **clingo** solver.

**Domain:** Symbolic AI & Logic  ·  **runnable:** yes (`pip install clingo`, pure-CPU, tiny problems)

## 1. What & Why

**Answer Set Programming (ASP)** is logic programming for *search*. You write a logic program whose **stable models** (a.k.a. *answer sets*) are exactly the solutions to your problem, then a solver enumerates them. Unlike Prolog you don't write a procedure that searches — you write a *specification of what a solution looks like*, and the grounder + solver find every model that satisfies it.

The mental shorthand is **Generate–Define–Test**:

```
1 { in(N) : node(N) } 3.        % GENERATE: pick 1–3 nodes
reachable(X) :- in(X).          % DEFINE:   derive consequences
:- in(X), in(Y), X < Y, bad(X,Y).   % TEST:    forbid bad combinations
```

**The problem it solves.** A huge class of hard problems is "find an assignment satisfying these constraints" — graph colouring, scheduling, planning, configuration, diagnosis, combinatorial optimization. ASP gives you a single declarative language for all of them, backed by a solver (clingo) built on the same conflict-driven (CDCL) technology as modern SAT solvers. You get SAT-level search power but speak in *rules and constraints over relations* instead of raw boolean clauses.

**What makes it different from SAT/Prolog/Datalog:**
- **Stable-model semantics** handles **non-monotonic** reasoning — defaults, exceptions, "assume X unless you can prove otherwise" — which plain SAT and Datalog can't express directly.
- **First-order rules with variables** are *grounded* to propositions automatically, so you write compact relational programs, not millions of clauses.
- **Built-in optimization** (`#minimize` / `#maximize`) finds not just *a* model but the *best* one.

**Reach for ASP when:** you have a combinatorial search / constraint / planning / configuration problem, you want all solutions (or the optimal one), and you value a clean declarative spec over hand-written search. **Skip it when:** the problem is numeric/continuous (use SMT or an MILP solver), purely a recursive *query* over data with no search (use [[datalog]]), or needs general computation and I/O (use [[swi-prolog]]).

## 2. Mental Model

> **ASP = "describe the solution, let the solver enumerate every world that fits."** A program isn't a query you run top-down; it's a set of constraints, and its *stable models* are the worlds consistent with those constraints.

Two ideas make it click:

**1. Generate–Define–Test.** Choice rules *generate* a search space, ordinary rules *define* derived facts, and integrity constraints (`:-` with empty head) *prune* the bad candidates. The solver searches this space with CDCL conflict learning.

```
        GENERATE                 TEST
   ┌──────────────────┐    ┌──────────────────┐
   │ 1 {assign(N,C):  │    │ :- edge(X,Y),    │
   │   color(C)} 1 :- │ →  │    assign(X,C),  │ →   ANSWER SETS
   │   node(N).       │    │    assign(Y,C).  │   (each = one valid
   └──────────────────┘    └──────────────────┘    full assignment)
     all colourings          drop the illegal ones
```

**2. Stable models, not first-derivation.** An answer set is a set of atoms that is **exactly self-justifying**: every atom in it has a supporting rule whose body holds, and nothing is in it "for free." This is what makes `not` (negation as failure) behave as a *default*: `flies(X) :- bird(X), not ab(X)` means "a bird flies in any model where we have no reason to call it abnormal." Add `ab(X) :- penguin(X)` and the penguin's stable model drops `flies`. Plain Datalog/SAT can't capture that closed-world, non-monotonic flip.

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **Atom / literal** | `assign(1,red)` is an atom; a literal is an atom or its (default) negation `not a`. Lowercase = constant/predicate, Uppercase = variable. |
| **Fact** | A rule with empty body: `node(1).` Always true. |
| **Rule** | `Head :- Body.` — derive Head when Body holds. Horn-like but with `not`. |
| **Integrity constraint** | A rule with **no head**: `:- a, b.` — forbid any model where the body holds. The primary "Test" device. |
| **Choice rule** | `lo { p(X) : q(X) } hi :- cond.` — non-deterministically include between `lo` and `hi` of the matching atoms. The "Generate" device. |
| **Grounding** | Replacing variables with all constant instances, turning the first-order program into a finite propositional one. Done by *gringo* before solving. |
| **Stable model / answer set** | A self-supporting set of atoms satisfying every rule — the program's *solutions*. A program can have 0, 1, or many. |
| **Negation as failure (`not`)** | `not a` holds when `a` is *not* in the model. Enables defaults & non-monotonic reasoning (closed-world). |
| **Aggregates** | `#count`, `#sum`, `#min`, `#max` inside rules/constraints, e.g. `:- #count{ N : pick(N) } > 3.` |
| **Optimization** | `#minimize { W,T : ... }` / `#maximize { ... }` — search for the model with the best total weight; clingo proves optimality. |
| **Disjunction** | `a ; b :- c.` — head disjunction (genuinely harder, Σᵖ₂); lets ASP encode problems beyond NP. |
| **`#show`** | Restrict printed output to chosen predicates, e.g. `#show assign/2.` |

## 4. Setup

**clingo** (the *gringo* grounder + *clasp* solver, from the [Potassco](https://potassco.org/) suite) ships as a pip wheel with full Python bindings — no separate binary needed for the API used here.

```bash
pip install clingo        # Python bindings + bundled solver
```

There's also a standalone CLI: write a program to `prog.lp` and run `clingo prog.lp 0` (the `0` means "all models"). The cell below installs the wheel if it's missing and defines a tiny `solve()` helper used by every example. clingo is small and CPU-only, so all cells run on a fresh kernel with no GPU, no network, and no API key.

In [ ]:
# Install clingo only if it isn't already present (small, CPU-only wheel).
try:
    import clingo
except ImportError:
    %pip install -q clingo
    import clingo

def solve(program, models=0):
    """Enumerate stable models of an ASP program.

    models=0 -> all answer sets; models=1 -> just the first found.
    Returns a list of answer sets, each a sorted list of shown atoms.
    """
    ctl = clingo.Control()
    ctl.configuration.solve.models = models      # 0 = enumerate every model
    ctl.add("base", [], program)                 # load the logic program
    ctl.ground([("base", [])])                   # variables -> propositions
    answer_sets = []
    with ctl.solve(yield_=True) as handle:
        for m in handle:
            answer_sets.append(sorted(str(a) for a in m.symbols(shown=True)))
    return answer_sets

print("clingo", ".".join(map(str, clingo.version())), "ready")

## 5. Worked Examples

### Example 1 — Graph 3-colouring (the canonical Generate–Test)

Colour every node of a graph so no edge joins two same-coloured nodes. The **choice rule** `1 { assign(N,C) : color(C) } 1` generates exactly one colour per node; the **integrity constraint** `:- edge(X,Y), assign(X,C), assign(Y,C)` throws out any colouring with a monochromatic edge. Each surviving stable model *is* a valid colouring — and we can ask the solver to enumerate them all.

In [ ]:
graph = """
node(1..6).
edge(1,2). edge(1,3). edge(2,4). edge(3,4). edge(4,5). edge(5,6). edge(2,6).
color(red). color(green). color(blue).

% GENERATE: each node gets exactly one colour
1 { assign(N,C) : color(C) } 1 :- node(N).

% TEST: forbid two adjacent nodes sharing a colour
:- edge(X,Y), assign(X,C), assign(Y,C).

#show assign/2.
"""

one = solve(graph, models=1)[0]      # first valid colouring
allc = solve(graph, models=0)        # every valid colouring

print("a valid 3-colouring:")
for a in one:
    print("   ", a)
print("total valid 3-colourings:", len(allc))

### Example 2 — Defaults & exceptions (non-monotonic reasoning)

This is what ASP does that SAT and [[datalog]] can't say directly. "A bird flies *unless* we have reason to believe it's abnormal." The rule `flies(X) :- bird(X), not ab(X)` uses **negation as failure**: `flies` is concluded in any model lacking a derivation of `ab`. Adding the exception `ab(X) :- penguin(X)` makes the penguin's stable model *retract* the default conclusion — adding knowledge removed a conclusion, the hallmark of **non-monotonic** reasoning.

In [ ]:
defaults = """
bird(tweety).  bird(opus).  penguin(opus).

% a bird flies UNLESS we can show it's abnormal (negation as failure)
flies(X) :- bird(X), not ab(X).
ab(X)    :- penguin(X).

#show flies/1.
"""

print("stable model:", solve(defaults))
# tweety flies (no reason to think otherwise); opus the penguin does NOT.

### Example 3 — Optimization with `#maximize` (maximum independent set)

ASP doesn't stop at *feasible* — `#minimize` / `#maximize` find the *best* model and clingo **proves optimality**. Here we pick the largest **independent set**: a subset of nodes with no edge between any two. `{ pick(N) } :- node(N)` generates an arbitrary subset, the constraint enforces independence, and `#maximize { 1,N : pick(N) }` drives toward the biggest set. clingo minimizes its internal cost vector, so a maximize objective shows up as a negative cost.

In [ ]:
mis = """
node(1..6).
edge(1,2). edge(1,3). edge(2,4). edge(3,4). edge(4,5). edge(5,6). edge(2,6).

{ pick(N) } :- node(N).                  % GENERATE: any subset of nodes
:- edge(X,Y), pick(X), pick(Y).          % TEST: independent (no adjacent pair)
#maximize { 1,N : pick(N) }.             % OPTIMIZE: as many nodes as possible

#show pick/1.
"""

ctl = clingo.Control()
ctl.configuration.solve.models = 0       # let it improve to the optimum
ctl.add("base", [], mis)
ctl.ground([("base", [])])

best = None
with ctl.solve(yield_=True) as handle:
    for m in handle:                     # each yielded model improves on the last
        best = (sorted(str(a) for a in m.symbols(shown=True)), list(m.cost))
    optimal = handle.get()

atoms, cost = best
print("optimal independent set:", atoms)
print("size:", -cost[0], "  clingo cost vector:", cost)
print("optimality proven:", optimal.satisfiable and not optimal.unknown)

## 6. Gotchas & Pitfalls

- **Grounding is the bottleneck, not solving.** clingo first *grounds* every variable to constants. A rule like `p(X,Y,Z) :- q(X), q(Y), q(Z)` over 1000 constants is a *billion* ground instances. If clingo hangs or eats memory, it's almost always grounding blow-up — add restricting domain predicates to the body, project away unused variables, and check the ground size with `clingo --text`.
- **Unsafe variables.** Every variable must appear in a *positive* body literal so the grounder knows its range. `p(X) :- not q(X).` is unsafe (X unbounded). Bind it: `p(X) :- dom(X), not q(X).` Same safety idea as [[datalog]].
- **`:-` is not "if".** A rule with an empty head, `:- body.`, is an **integrity constraint** ("never let body hold"), not an implication. Forgetting the head turns a definition into a constraint and silently changes meaning.
- **Stable ≠ classical models.** `p :- not p.` has **no** stable model (it's inconsistent), even though it has a classical model. And programs can have *zero* answer sets — "no solution" is a normal, meaningful outcome, not an error.
- **`not` is closed-world & default.** `not a` means "`a` is not derivable in this model," not "`a` is false in reality." Don't read it as classical negation. For classical negation you need `-a`, which is a different beast.
- **Disjunction is not choice.** `a ; b.` (disjunctive head) means *minimal* models containing a or b and jumps to the second level of the polynomial hierarchy (harder). For "pick some of these," use a **choice rule** `{ a; b }` instead — far cheaper and usually what you meant.
- **Counting all models is expensive.** `models=0` enumerates *every* answer set; for "is it satisfiable?" or "give me one," use `models=1`. Enumerating can be exponential.
- **Multi-shot vs one-shot.** Re-grounding a big program every call is wasteful. For planning with growing horizons, use clingo's *multi-shot* API (`#program` parts + `ctl.ground` with parameters) instead of rebuilding `Control` each time.

## 7. When to Use vs Alternatives

| Option | Sweet spot | Trade-off vs ASP |
|--------|-----------|------------------|
| **ASP (clingo)** | Combinatorial search, planning, configuration, diagnosis, KR with defaults; all-models & optimization | Declarative + non-monotonic + optimization in one language, NP/Σᵖ₂ power — but **discrete only**, grounding can blow up, no native floats/continuous math. |
| **SAT solvers** | Pure boolean satisfiability at massive scale | Fastest on raw CNF, but you hand-encode everything; no variables, no rules, no defaults, no recursion. ASP *compiles down to* this. |
| **[[z3-smt]] (SMT)** | Constraints over ints, **reals**, arrays, bitvectors; verification | Handles continuous/arithmetic theories ASP can't; weaker at "enumerate all models" and default reasoning. Use SMT when numbers/theories dominate. |
| **MILP (Gurobi/CBC)** | Large-scale numeric optimization, linear/continuous objectives | Industrial optimization over reals; ASP wins on logical/combinatorial structure and qualitative rules, MILP wins on big linear-numeric models. |
| **[[minizinc]] (CP)** | Constraint programming: scheduling, rostering, finite-domain integers | Rich global constraints & integer arithmetic; ASP wins on recursion, defaults, and KR; CP often wins on heavy arithmetic/scheduling. |
| **[[datalog]]** | Recursive *queries* over relational data, deductive DBs | Decidable, monotonic, no search — strictly a query engine. Use it when you're *deriving* facts, not *searching* for a model. |
| **[[swi-prolog]] (Prolog)** | General logic programming, terms, I/O, top-down search | Turing-complete with side effects; but you control the search and termination. ASP is purely declarative model-finding with a complete solver. |
| **[[problog]]** | *Probabilistic* logic / inference over uncertain facts | Adds probabilities to logic; ASP is about hard constraints & optimization, not probability mass. |

**Rule of thumb:** discrete *search/constraint/planning* problem where you want a clean spec and all-or-optimal solutions → **ASP**. Continuous or arithmetic-heavy → SMT/MILP/CP. Just *deriving* facts with no search → Datalog. Need probabilities → ProbLog.

## 8. Resources

- **Potassco — the clingo home** (downloads, the Python API guide, and `clingo`/`gringo`/`clasp` docs): <https://potassco.org/>
- **clingo Python API reference** (the `Control`, grounding, and solve interfaces used above): <https://potassco.org/clingo/python-api/current/clingo/>
- **"Answer Set Solving in Practice"** — Gebser, Kaminski, Kaufmann & Schaub; the definitive synthesis-lecture book: <https://potassco.org/book/>
- **"Modeling and Language Extensions"** (Gebser et al., AI Magazine) — concise, readable intro to the modelling methodology: <https://ojs.aaai.org/aimagazine/index.php/aimagazine/article/view/2599>
- **A User's Guide to gringo, clasp, clingo, and iclingo** (the canonical syntax & options reference): <https://github.com/potassco/guide/releases>
- **Wikipedia: Answer set programming** (solid overview of stable-model semantics and history): <https://en.wikipedia.org/wiki/Answer_set_programming>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def is_stable(rules, model):
    """True when `model` is a stable model (answer set) of `rules`."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE